# Market data basics

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

This notebook is a software tutorial using bundled example bars or a small deterministic fixture. It is not a trading recommendation or a research performance claim.

We will read the small bundled US100 minute-bar sample, check its time and price fields, then plot a short session. The tab-separated source preserves broker-style date and time columns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
path = ROOT / "data/examples/US100.cash_M1_202412300105_202501020334.csv"
bars = pd.read_csv(path, sep="\t")
bars["timestamp"] = pd.to_datetime(bars["<DATE>"] + " " + bars["<TIME>"])
bars = bars.set_index("timestamp").rename(
    columns={"<OPEN>": "open", "<HIGH>": "high", "<LOW>": "low", "<CLOSE>": "close"}
)
print(f"{len(bars):,} bars from {bars.index.min()} to {bars.index.max()}")
bars[["open", "high", "low", "close"]].head()

Figure: US100 close in the first 180 M1 bars, with source timestamps on the x-axis and index points on the y-axis.

In [ ]:
sample = bars.iloc[:180]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sample.index, sample["close"], label="US100 close", color="#2369a8")
ax.set(
    title="Bundled US100 M1 sample: first 180 bars",
    ylabel="Index points",
    xlabel="Source timestamp",
)
ax.legend()
fig.autofmt_xdate()
plt.show()

## Validate the source before plotting

A chart can look plausible even when rows are duplicated, prices are missing, or timestamps are out of order. Check these properties before handing bars to a feature pipeline.


In [ ]:
price_columns = ["open", "high", "low", "close"]
quality = {
    "monotonic_time": bool(bars.index.is_monotonic_increasing),
    "duplicate_timestamps": int(bars.index.duplicated().sum()),
    "missing_prices": int(bars[price_columns].isna().sum().sum()),
    "high_below_low": int((bars["high"] < bars["low"]).sum()),
}
print(quality)
assert quality["monotonic_time"] and quality["duplicate_timestamps"] == 0
assert quality["missing_prices"] == 0 and quality["high_below_low"] == 0

## Derive a simple bar return

The percentage change uses the previous observed close. It is a descriptive transformation, not a predictive feature claim; production features must still respect split boundaries and training-only normalization.


In [ ]:
bars["close_return"] = bars["close"].pct_change()
returns = bars["close_return"].dropna()
print({"observations": len(returns), "mean": returns.mean(), "std": returns.std()})
assert len(returns) == len(bars) - 1

## Interpret the output

The timestamps and OHLC values are the inputs to later transforms. The sample is intentionally small; production ingestion additionally validates configured sources, splits, and cache identity.

See [market-data API](../../api/domains/data), [prepare market data](../../how-to/prepare_market_data), and [market data and features](../../framework/market_data_and_features).